# 05 — Modelagem Baseline e Tuning de Hiperparâmetros

**Objetivo:** Treinar e comparar modelos para prever óbito por SRAG 2023. Selecionar o melhor pipeline pelo AUC-PR em validação cruzada temporal.

**Inputs:**
- `data/processed/X_train.parquet` — 145.407 casos × 45 features
- `data/processed/X_test.parquet`  — 103.072 casos × 45 features
- `data/processed/y_train.parquet` — target binário (1=Óbito, 0=Cura)
- `data/processed/y_test.parquet`

**Outputs:**
- `models/best_model.joblib` — pipeline sklearn serializado
- `reports/figures/05_modelagem/{01_pr_curves, 02_roc_curves}.png`

**Decisões-chave:**
1. **Métrica primária: AUC-PR** — óbito é a classe rara (cerca de 10%); AUC-PR é mais sensível ao desempenho na classe positiva que AUC-ROC.
2. **Split temporal** — 1º semestre (treino) / 2º semestre (teste). Simula uso real; sem data leakage temporal.
3. **Tuning: Optuna (TPE)** — busca bayesiana eficiente; 3-fold CV em subsample para cada trial, 5-fold no treino completo para estimativa final honesta.
4. **Imbalance: class_weight / scale_pos_weight** — preferido sobre SMOTE para dados tabulares com alta dimensionalidade. SMOTE é comparado no notebook 06.

| # | Modelo | Papel |
|---|---|---|
| 1 | DummyClassifier | Piso mínimo de referência |
| 2 | Regressão Logística | Baseline linear interpretável |
| 3 | Random Forest (default e tuned) | Bagging robusto a outliers |
| 4 | XGBoost (default e tuned) | Boosting, alta acurácia |
| 5 | LightGBM (default e tuned) | Boosting eficiente (leaf-wise) |

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

sys.path.insert(0, str(Path("..").resolve()))

from maxpar_srag import config
from maxpar_srag.features import (
    BINARY_FEATURES, MISS_FEATURES, NUMERIC_FEATURES, OHE_FEATURES, ORDINAL_FEATURES,
)
from maxpar_srag.models import (
    FIGURES_DIR_05,
    build_dummy, build_lgbm, build_logistic, build_rf, build_xgb,
    evaluate_cv,
    plot_pr_curves, plot_roc_curves,
    save_model,
    tune_lgbm, tune_rf, tune_xgb,
)

mpl.rcParams.update({"figure.dpi": 100, "figure.facecolor": "white"})
FIGURES_DIR_05.mkdir(parents=True, exist_ok=True)

SEED = config.RANDOM_STATE
print(f"RANDOM_STATE = {SEED}")

In [ ]:
X_train = pd.read_parquet(config.DATA_PROCESSED / "X_train.parquet")
X_test  = pd.read_parquet(config.DATA_PROCESSED / "X_test.parquet")
y_train = pd.read_parquet(config.DATA_PROCESSED / "y_train.parquet")["target"]
y_test  = pd.read_parquet(config.DATA_PROCESSED / "y_test.parquet")["target"]

feature_cols = X_train.columns.tolist()
results_cv: dict[str, dict] = {}  # acumula {nome: {metric: (mean, std)}}

print(f"X_train : {X_train.shape}  | óbito = {y_train.mean():.2%}")
print(f"X_test  : {X_test.shape}   | óbito = {y_test.mean():.2%}")
print(f"Features: {len(feature_cols)} colunas")
print(f"  numéricas  : {NUMERIC_FEATURES}")
print(f"  binárias   : {len(BINARY_FEATURES)} colunas")
print(f"  _MISS      : {len(MISS_FEATURES)} indicadores")
print(f"  ordinais   : {ORDINAL_FEATURES}")
print(f"  OHE        : {OHE_FEATURES}")

## 1. Baseline Ingênuo — DummyClassifier

O `DummyClassifier(strategy='stratified')` amostra as classes na proporção do treino, ignorando totalmente as features.

**Expectativa:** AUC-PR próximo da prevalência de óbito (cerca de 10%); AUC-ROC próximo de 0,50.

Qualquer modelo real deve superar estas marcas com folga. Se não superar, há problema no pipeline ou nas features.

In [ ]:
print("=== 1. DummyClassifier (estratificado) — CV 5-Fold ===")
pipe_dummy = build_dummy(feature_cols)
results_cv["Dummy"] = evaluate_cv(pipe_dummy, X_train, y_train)

## 2. Regressão Logística — Baseline Interpretável

A regressão logística é o único modelo desta lista com coeficientes legíveis como log-odds. Serve de baseline linear: se os modelos não lineares não superarem a LR com boa margem, a relação entre o target e as features tem forte componente linear.

A opção `class_weight='balanced'` é necessária porque a taxa de óbito é de cerca de 10%, fazendo com que o gradiente médio seja dominado pela classe Cura sem ponderação. O parâmetro `balanced` atribui peso inversamente proporcional à frequência de classe, melhorando o recall sem reamostragem.

O valor `C=0.1` corresponde a regularização L2 moderada, adequada para dados do SIVEP-Gripe que apresentam ruído de preenchimento e features correlacionadas (comorbidades e `IDADE_ANOS`). A regularização reduz a influência de features com sinal fraco.

O preprocessor (imputação e scaler) é ajustado apenas no treino de cada fold, sem contaminação do conjunto de teste.

In [ ]:
print("=== 2. Regressão Logística (C=0.1, balanced) — CV 5-Fold ===")
pipe_lr = build_logistic(feature_cols, C=0.1)
results_cv["Logística"] = evaluate_cv(pipe_lr, X_train, y_train)

## 3. Random Forest

O Random Forest é um ensemble de árvores com bootstrap (bagging). Suas principais vantagens para a base SRAG 2023 são robustez a outliers e ao ruído de imputação em comorbidades, além da capacidade de capturar interações não lineares entre variáveis como `IDADE_ANOS`, `IMUNODEPRE` e `SUPORT_VEN_ORD`.

A opção `class_weight='balanced_subsample'` pondera as classes dentro de cada bootstrap sample, sendo mais robusta que o `'balanced'` global em datasets grandes.

O SMOTE não é aplicado aqui porque exige imputação prévia à geração de amostras sintéticas, o que complica a inserção correta dentro de um pipeline sklearn. Essa comparação é feita no notebook 06.

In [ ]:
print("=== 3a. Random Forest (default) — CV 5-Fold ===")
pipe_rf_default = build_rf(feature_cols)
results_cv["RF (default)"] = evaluate_cv(pipe_rf_default, X_train, y_train)

### 3.1 Tuning de Hiperparâmetros — Optuna (TPE, 50 trials)

**Estratégia:**
- **Algoritmo:** TPE (Tree-structured Parzen Estimator) — aprende a distribuição de hiperparâmetros promissores a partir das tentativas anteriores. Mais eficiente que grid/random search em espaços de alta dimensão.
- **CV interna:** 3 folds (em vez dos 5 da avaliação final) — preserva a ordenação relativa dos hiperparâmetros com menor custo por trial.
- **Subsample:** 60 mil linhas (cerca de 41% do treino) — reduz o custo de cada fit; a correlação de desempenho em subconjuntos é alta para RF.
- **Hiperparâmetros buscados:** `n_estimators`, `max_depth`, `min_samples_leaf`, `max_features`.
- **Após tuning:** avaliação final com 5-fold CV no treino completo (145 mil casos) para estimativa honesta.

In [ ]:
%%time
print("=== Optuna RF (50 trials, subsample=60k) ===")
study_rf = tune_rf(X_train, y_train, feature_cols, n_trials=50, n_sample=60_000)
print(f"\nMelhores hiperparâmetros RF:")
for k, v in study_rf.best_params.items():
    print(f"  {k}: {v}")
print(f"\nMelhor AUC-PR (3-fold, 60k): {study_rf.best_value:.4f}")

In [ ]:
print("=== 3b. Random Forest (tuned) — CV 5-Fold (treino completo) ===")
pipe_rf_tuned = build_rf(feature_cols, **study_rf.best_params)
results_cv["RF (tuned)"] = evaluate_cv(pipe_rf_tuned, X_train, y_train)

## 4. XGBoost

O XGBoost é um algoritmo de gradient boosting com gradientes de segunda ordem (hessiana), historicamente competitivo em tarefas com dados tabulares. Seu crescimento level-wise (por nível de profundidade) é mais estável, embora potencialmente mais lento que o LightGBM.

O parâmetro `scale_pos_weight = 9` equivale ao `class_weight='balanced'`: calcula-se como a razão entre negativos e positivos, aproximadamente 90 mil / 10 mil = 9. O Optuna explorará valores entre 5 e 15 para otimizar.

A métrica de avaliação interna `eval_metric='aucpr'` alinha o critério interno (usado em early stopping quando ativado) com a métrica primária do projeto.

In [ ]:
print("=== 4a. XGBoost (default) — CV 5-Fold ===")
pipe_xgb_default = build_xgb(feature_cols)
results_cv["XGB (default)"] = evaluate_cv(pipe_xgb_default, X_train, y_train)

In [ ]:
%%time
print("=== Optuna XGBoost (80 trials, subsample=100k) ===")
study_xgb = tune_xgb(X_train, y_train, feature_cols, n_trials=80, n_sample=100_000)
print(f"\nMelhores hiperparâmetros XGB:")
for k, v in study_xgb.best_params.items():
    print(f"  {k}: {v}")
print(f"\nMelhor AUC-PR (3-fold, 100k): {study_xgb.best_value:.4f}")

In [ ]:
print("=== 4b. XGBoost (tuned) — CV 5-Fold (treino completo) ===")
pipe_xgb_tuned = build_xgb(feature_cols, **study_xgb.best_params)
results_cv["XGB (tuned)"] = evaluate_cv(pipe_xgb_tuned, X_train, y_train)

## 5. LightGBM

O LightGBM é uma implementação de gradient boosting com crescimento leaf-wise, que escolhe a folha de maior ganho em vez de expandir o nível inteiro. As principais vantagens são velocidade (tipicamente 10 a 30 vezes mais rápido que XGBoost em datasets grandes, permitindo mais trials de Optuna no mesmo tempo), menor uso de memória por trabalhar com histogramas de features e o parâmetro `min_child_samples`, que previne overfitting em folhas com poucos casos — relevante para a classe óbito, que representa cerca de 10% do treino.

O parâmetro `bagging_freq=1` é necessário para ativar o subsampling de linhas quando `subsample < 1.0`.

Espera-se que LightGBM e XGBoost apresentem AUC-PR próximo; a diferença costuma ficar dentro do desvio-padrão do CV.

In [ ]:
print("=== 5a. LightGBM (default) — CV 5-Fold ===")
pipe_lgbm_default = build_lgbm(feature_cols)
results_cv["LGBM (default)"] = evaluate_cv(pipe_lgbm_default, X_train, y_train)

In [ ]:
%%time
print("=== Optuna LightGBM (80 trials, subsample=100k) ===")
study_lgbm = tune_lgbm(X_train, y_train, feature_cols, n_trials=80, n_sample=100_000)
print(f"\nMelhores hiperparâmetros LGBM:")
for k, v in study_lgbm.best_params.items():
    print(f"  {k}: {v}")
print(f"\nMelhor AUC-PR (3-fold, 100k): {study_lgbm.best_value:.4f}")

In [ ]:
print("=== 5b. LightGBM (tuned) — CV 5-Fold (treino completo) ===")
pipe_lgbm_tuned = build_lgbm(feature_cols, **study_lgbm.best_params)
results_cv["LGBM (tuned)"] = evaluate_cv(pipe_lgbm_tuned, X_train, y_train)

## 6. Comparação Final — Seleção do Melhor Modelo

Todos os modelos são comparados pelo **AUC-PR médio em 5-fold CV** no conjunto de treino.

**Critérios de desempate (em ordem):**
1. **AUC-PR** (primária): detecta desempenho na classe rara
2. **Recall (óbito)**: sensibilidade na classe positiva — custo de falso negativo alto em saúde pública
3. **AUC-ROC**: discriminação geral
4. **F1**: harmônica de precisão e recall ao threshold padrão (0.5)

O modelo selecionado é retreinado no treino completo (sem CV) e avaliado honestamente no conjunto de teste temporal.

In [ ]:
rows = []
for name, metrics in results_cv.items():
    rows.append({
        "Modelo": name,
        "AUC-PR": f"{metrics['ap'][0]:.4f} \u00b1{metrics['ap'][1]:.4f}",
        "AUC-ROC": f"{metrics['roc_auc'][0]:.4f} \u00b1{metrics['roc_auc'][1]:.4f}",
        "F1": f"{metrics['f1'][0]:.4f} \u00b1{metrics['f1'][1]:.4f}",
        "Recall (\u00f3bito)": f"{metrics['recall'][0]:.4f} \u00b1{metrics['recall'][1]:.4f}",
        "_ap_raw": metrics["ap"][0],
    })

df_comp = (
    pd.DataFrame(rows)
    .set_index("Modelo")
    .sort_values("_ap_raw", ascending=False)
)

print("Ranking por AUC-PR (CV 5-Fold, treino completo):")
display(df_comp.drop(columns=["_ap_raw"]))

best_name = df_comp.index[0]
best_ap_cv = df_comp["_ap_raw"].iloc[0]
print(f"\n>>> Melhor modelo: {best_name} (AUC-PR CV = {best_ap_cv:.4f}) <<<")

## 7. Avaliação no Conjunto de Teste (Jul–Dez 2023)

Os modelos tuned são retreinados no treino completo (145 mil casos) e avaliados no teste temporal (103 mil casos). Como os hiperparâmetros foram obtidos sem exposição ao conjunto de teste, a avaliação é honesta.

**Curva PR e ROC:**
- **PR** (primária): a área sob a curva é a Average Precision. Mais informativa para classes desbalanceadas — mostra o trade-off precisão-recall diretamente na classe de interesse.
- **ROC**: mais utilizada em comunicação clínica (sensibilidade e especificidade). Complementa a curva PR.

São plotados apenas os modelos tuned (e Dummy e LR como referência) para clareza visual.

In [ ]:
# Construir e treinar todos os modelos finais no treino completo
_final_builds = {
    "Dummy": (build_dummy, {}),
    "Logística": (build_logistic, {"C": 0.1}),
    "RF (tuned)": (build_rf, study_rf.best_params),
    "XGB (tuned)": (build_xgb, study_xgb.best_params),
    "LGBM (tuned)": (build_lgbm, study_lgbm.best_params),
}

fitted_final: dict = {}
for name, (build_fn, kwargs) in _final_builds.items():
    print(f"  Fitting {name}...", end=" ", flush=True)
    p = build_fn(feature_cols, **kwargs)
    p.fit(X_train, y_train)
    fitted_final[name] = p
    print("OK")

# Mapear best_name (que pode ser "RF (default)" etc.) para a versão tuned em fitted_final
_family_to_tuned = {
    "Dummy": "Dummy",
    "Logística": "Logística",
    "RF (default)": "RF (tuned)",
    "RF (tuned)": "RF (tuned)",
    "XGB (default)": "XGB (tuned)",
    "XGB (tuned)": "XGB (tuned)",
    "LGBM (default)": "LGBM (tuned)",
    "LGBM (tuned)": "LGBM (tuned)",
}
_best_key = _family_to_tuned.get(best_name, "LGBM (tuned)")
best_pipeline = fitted_final[_best_key]
print(f"\nMelhor pipeline para serialização: {_best_key}")

In [ ]:
fig_pr = plot_pr_curves(
    fitted_final, X_test, y_test,
    save_path=FIGURES_DIR_05 / "01_pr_curves.png",
)
plt.show()

In [ ]:
fig_roc = plot_roc_curves(
    fitted_final, X_test, y_test,
    save_path=FIGURES_DIR_05 / "02_roc_curves.png",
)
plt.show()

In [ ]:
from maxpar_srag.models import save_model
from sklearn.metrics import average_precision_score, roc_auc_score

# M\u00e9tricas no teste para o melhor pipeline
proba_best = best_pipeline.predict_proba(X_test)[:, 1]
ap_test = average_precision_score(y_test, proba_best)
auc_test = roc_auc_score(y_test, proba_best)

print(f"=== M\u00e9tricas no conjunto de teste ({_best_key}) ===")
print(f"  AUC-PR  (teste): {ap_test:.4f}")
print(f"  AUC-ROC (teste): {auc_test:.4f}")
print(f"  AUC-PR  (CV)  : {best_ap_cv:.4f}  [refer\u00eancia]")

model_path = save_model(best_pipeline, "best_model")
print(f"\nFiguras salvas em: {FIGURES_DIR_05}")
print("  01_pr_curves.png")
print("  02_roc_curves.png")

## Síntese

### O que foi feito
1. **8 configurações de modelo** avaliadas por CV 5-fold no treino temporal (Jan–Jun 2023): Dummy, Logística, RF, XGBoost e LightGBM — cada um com versão default e tuned.
2. **Tuning Optuna (TPE):** RF com 50 trials em 60 mil amostras; XGBoost e LightGBM com 80 trials em 100 mil amostras. CV 3-fold por trial para eficiência.
3. **Melhor modelo selecionado** por AUC-PR e avaliado honestamente no conjunto de teste temporal (Jul–Dez 2023).
4. **Pipeline serializado** em `models/best_model.joblib` — inclui preprocessor (imputação, OHE e scaler) e classificador.

### Resultados no conjunto de teste (Jul–Dez 2023)
- **Modelo selecionado:** XGBoost (tuned) — AUC-PR **0,5531** · AUC-ROC **0,9055**
- Baseline aleatório: AUC-PR próximo de 0,10 (prevalência) — o modelo alcança 5,5 vezes o baseline aleatório
- AUC-ROC de 0,90 confirma excelente separação global entre óbito e cura
- A diferença entre AUC-PR (0,55) e AUC-ROC (0,90) é esperada com 10% de positivos: AUC-ROC não penaliza a grande quantidade de negativos

### Decisão sobre desbalanceamento
Utilizou-se `class_weight='balanced'` (LR, RF, LightGBM) e `scale_pos_weight` (XGBoost) como estratégia de ponderação de gradiente. O notebook 06 compara essa abordagem com threshold tuning e SMOTE.

### Próxima fase — Notebook 06
- **Threshold tuning:** comparar threshold 0,5 com Youden e recall mínimo clínico (sensibilidade >= 70%)
- **SMOTE vs class_weight:** impacto no AUC-PR e recall no conjunto de teste
- **SHAP:** importância global (beeswarm) e individual (force plot)
- **Análise de subgrupo:** desempenho por UF, faixa etária e raça
- **Calibração:** reliability diagram e Brier score